In [ ]:
import math
import numpy as np
from scipy.sparse import csr_matrix
from collections import defaultdict
import os

# Load the file
data_path = 'user_movie_rating.npy'
data = np.load(data_path)
print("Loaded data sample:", data[:5])

# Extract users and movies from data
users = data[:, 0]
movies = data[:, 1]

# Create a sparse matrix for movie-user ratings
movie_user_matrix = csr_matrix((np.ones_like(users), (movies - 1, users - 1)))

# Check the shape
print("Sparse matrix shape (movies x users):", movie_user_matrix.shape)
# Check specific value (e.g., user 1 rating movie 30)
print("User 1 rating movie 30:", movie_user_matrix[29, 0])

def generate_minhash_signature_updated(sparse_matrix, num_permutations):
    """
    Compute MinHash signatures using permutations.

    Parameters:
        sparse_matrix (csr_matrix): Sparse movie-user matrix where rows represent movies, columns represent users.
        num_permutations (int): Number of MinHash permutations (hash functions) to apply.

    Returns:
        np.ndarray: A 2D array of shape (num_permutations, num_users) where each row
                    corresponds to a permutation and each column to a user's MinHash value.
    """
    num_movies, num_users = sparse_matrix.shape

    # initialize signature matrix with +infinity
    signatures = np.full((num_permutations, num_users), np.inf)

    # generate random hash functions (permutations) for rows
    hash_functions = [np.random.permutation(num_movies) for _ in range(num_permutations)]

    # Precompute inverse hash functions for fast lookup
    inverse_hash_functions = [np.argsort(h) for h in hash_functions]

    # iterate over each row (movie)
    for row_idx in range(num_movies):
        row = sparse_matrix[row_idx, :]  # access the sparse row
        if row.nnz == 0:  # Skip empty rows
            continue

        # Get the users (columns) that rated this movie
        user_indices = row.indices

        for perm_idx in range(num_permutations):  # for each hash function (permutation)
            # Retrieve the position of row_idx in the permutation
            hash_value = inverse_hash_functions[perm_idx][row_idx]
            # Update signatures for all users who rated this movie
            signatures[perm_idx, user_indices] = np.minimum(signatures[perm_idx, user_indices], hash_value)

    return signatures

def lsh(signature_matrix, num_bands, rows_per_band, num_buckets=2**16):
    """
    Perform Locality Sensitive Hashing (LSH) on a MinHash signature matrix.

    Parameters:
        signature_matrix (np.ndarray): MinHash signature matrix of shape (n, m),
                                        where n = number of hash functions (total rows),
                                        and m = number of users (columns).
        num_bands (int): Number of bands to divide the matrix into.
        rows_per_band (int): Number of rows per band (b * r must equal n).
        num_buckets (int): Number of buckets for hashing (large value to reduce collisions).

    Returns:
        set: A set of candidate column pairs (as tuples) identified by LSH.
    """
    num_hashes, num_users = signature_matrix.shape
    assert num_bands * rows_per_band == num_hashes, \
        "Number of bands * rows per band must equal total hash functions"

    candidate_pairs = set()

    # Process each band
    for band_idx in range(num_bands):
        # Extract rows corresponding to the current band
        start_row = band_idx * rows_per_band
        end_row = start_row + rows_per_band
        band = signature_matrix[start_row:end_row, :]

        # Hash columns (users) in the band into buckets
        buckets = defaultdict(list)
        for user_idx in range(num_users):
            band_hash = hash(tuple(band[:, user_idx])) % num_buckets
            buckets[band_hash].append(user_idx)

        # Add candidate pairs for columns hashed to the same bucket
        for bucket in buckets.values():
            if len(bucket) > 1:
                # Add all unique pairs from this bucket
                for i in range(len(bucket)):
                    for j in range(i + 1, len(bucket)):
                        candidate_pairs.add((bucket[i], bucket[j]))

    print(f"Generated {len(candidate_pairs)} candidate pairs")  # Debugging line
    return candidate_pairs

def compute_jaccard_similarity(sparse_matrix, candidate_pairs, threshold=0.5, output_file="output1.txt"):
    """
    Compute Jaccard Similarity for candidate pairs and write the results to a file.

    Parameters:
        sparse_matrix (csr_matrix): Sparse movie-user matrix where rows represent movies, columns represent users.
        candidate_pairs (set): Set of candidate pairs as tuples (u1, u2).
        threshold (float): The minimum Jaccard Similarity to include a pair.
        output_file (str): File path to write the results.
    """
    print(f"Number of candidate pairs: {len(candidate_pairs)}")  # Debugging line
    
    # Open output file and write results
    with open(output_file, "w") as file:
        for u1, u2 in candidate_pairs:
            # Get the sets of movies rated by user u1 and user u2
            movies_u1 = set(sparse_matrix[:, u1].indices)
            movies_u2 = set(sparse_matrix[:, u2].indices)

            # Compute Jaccard Similarity
            intersection_size = len(movies_u1 & movies_u2)  # Intersection of two sets
            union_size = len(movies_u1 | movies_u2)        # Union of two sets

            if union_size > 0:  # Avoid division by zero
                jaccard_similarity = intersection_size / union_size
                # Check if the similarity meets the threshold
                if jaccard_similarity > threshold:
                    # Write to the file in the form "u1,u2"
                    file.write(f"{u1},{u2}\n")
                    print(f"Writing: {u1},{u2} with similarity {jaccard_similarity}")  # Debugging line

# Start of the main process
# Step 1: Generate MinHash signatures
signatures = generate_minhash_signature_updated(movie_user_matrix, 100)
print("MinHash signature matrix shape:", signatures.shape)

# Step 2: Apply LSH to find candidate pairs
candidate_pairs = lsh(signatures, 5, 20, num_buckets=2**16)
print("Number of candidate pairs:", len(candidate_pairs))

# Step 3: Compute Jaccard Similarity and write results to a file
output_file = "output1.txt"
compute_jaccard_similarity(movie_user_matrix, candidate_pairs, threshold=0.5, output_file=output_file)

# Check the current working directory to confirm file writing location
print(f"Current working directory: {os.getcwd()}")


Loaded data sample: [[  1  30   3]
 [  1 157   3]
 [  1 173   4]
 [  1 175   5]
 [  1 191   2]]
MinHash signature matrix shape: (100, 103703)
Generated 410255 candidate pairs
Number of candidate pairs: 410255
Number of candidate pairs: 410255
Writing: 18698,94153 with similarity 1.0
Writing: 20210,61985 with similarity 1.0
Writing: 9372,23191 with similarity 1.0
Writing: 31060,49111 with similarity 1.0
Writing: 19774,58422 with similarity 1.0
Writing: 59865,103365 with similarity 1.0
Writing: 4897,77341 with similarity 1.0
Writing: 17632,94358 with similarity 1.0
Writing: 34249,103471 with similarity 1.0
Writing: 25044,95818 with similarity 1.0
Writing: 21474,58008 with similarity 1.0
Writing: 42704,47587 with similarity 1.0
Writing: 16503,44788 with similarity 1.0
Writing: 9592,72923 with similarity 1.0
Writing: 4526,79091 with similarity 1.0
Writing: 53380,72467 with similarity 1.0
Writing: 49092,75519 with similarity 1.0
Writing: 4799,77937 with similarity 1.0
Writing: 44197,66293 w

In [1]:
import numpy as np
import random
from scipy.sparse import coo_matrix
from collections import defaultdict

# Step 1: Load Data (for .npy file)
def load_data(file_path):
    # Load the user-movie rating data from .npy file
    data = np.load(file_path)  # Use np.load for .npy files
    return data

# Step 2: Create a Binary Sparse Matrix (User x Movie)
def create_binary_matrix(data, num_users, num_movies):
    # Creating a binary sparse matrix where rows represent users and columns represent movies
    rows = data[:, 0] - 1  # User IDs (0-indexed)
    cols = data[:, 1] - 1  # Movie IDs (0-indexed)
    values = np.ones(len(data))  # The ratings themselves are irrelevant, just 1 if rated
    matrix = coo_matrix((values, (rows, cols)), shape=(num_users, num_movies))
    return matrix

# Step 3: Generate Minhash Signatures (Hashing Function)
def minhash_signatures(matrix, num_hashes):
    num_users, num_movies = matrix.shape
    permutations = [np.random.permutation(num_movies) for _ in range(num_hashes)]  # Random permutations for hashing
    signatures = np.full((num_hashes, num_users), np.inf)  # Initialize signatures with inf values
    
    for user in range(num_users):
        rated_movies = set(matrix.getrow(user).nonzero()[1])  # Get movies rated by the user using getrow()
        for h, perm in enumerate(permutations):
            for movie in rated_movies:
                signatures[h, user] = min(signatures[h, user], perm[movie])
                break  # Move to the next hash function once the first rated movie is found
    return signatures

# Step 4: Apply Banding Technique and Find Candidate Pairs
def banding(signatures, num_bands, num_rows):
    b = num_bands
    r = num_rows
    candidates = set()  # Set to store candidate pairs
    num_hashes = signatures.shape[0]
    
    # Split the signatures into bands of rows
    for i in range(b):
        band = signatures[i * r: (i + 1) * r, :]  # Take r rows for the band
        band_hash = defaultdict(list)  # Dictionary to hash rows into buckets
        
        # Hash each user's signature into the band and group users with the same hash value
        for user in range(band.shape[1]):
            band_key = tuple(band[:, user])  # Hash key for the band
            band_hash[band_key].append(user)
        
        # If more than one user falls into the same bucket, consider them as candidate pairs
        for users in band_hash.values():
            if len(users) > 1:
                for u1 in range(len(users)):
                    for u2 in range(u1 + 1, len(users)):
                        candidates.add((users[u1], users[u2]))
    return candidates

# Step 5: Calculate Jaccard Similarity
def jaccard_similarity(user1_movies, user2_movies):
    # Calculate the Jaccard Similarity between two users based on rated movies
    intersection = len(user1_movies & user2_movies)
    union = len(user1_movies | user2_movies)
    return intersection / union if union > 0 else 0  # Prevent division by zero

# Step 6: Main Function to Find Similar Users Using LSH
def find_similar_users(file_path, num_hashes, num_bands, num_rows, threshold, output_file):
    # Load the data
    data = load_data(file_path)
    num_users = max(data[:, 0])  # Maximum user ID
    num_movies = max(data[:, 1])  # Maximum movie ID
    
    # Step 1: Create a binary sparse matrix
    matrix = create_binary_matrix(data, num_users, num_movies)
    
    # Step 2: Generate Minhash signatures
    signatures = minhash_signatures(matrix, num_hashes)
    
    # Step 3: Apply banding technique to find candidate pairs
    candidates = banding(signatures, num_bands, num_rows)
    
    # Step 4: Open the output file and write results
    with open(output_file, 'w') as f:
        for u1, u2 in candidates:
            user1_movies = set(matrix.getrow(u1).nonzero()[1])  # Movies rated by user 1
            user2_movies = set(matrix.getrow(u2).nonzero()[1])  # Movies rated by user 2
            
            # Calculate Jaccard Similarity for candidate pair
            js = jaccard_similarity(user1_movies, user2_movies)
            if js > threshold:
                # If similarity exceeds threshold, write the pair to the file
                f.write(f"{u1+1},{u2+1}\n")

# Step 7: Example usage of the LSH algorithm
if __name__ == "__main__":
    random.seed(42)  # Set a random seed for reproducibility
    
    # You can adjust the following parameters:
    num_hashes = 100  # Number of hash functions (Minhash signatures)
    num_bands = 5  # Number of bands
    num_rows = 20  # Number of rows per band
    threshold = 0.5  # Jaccard similarity threshold
    output_file = 'vaku.txt'  # File to save the output
    
    # Find similar users with the LSH algorithm
    find_similar_users('user_movie_rating.npy', num_hashes, num_bands, num_rows, threshold, output_file)


KeyboardInterrupt: 